### Data Ingestion

In [42]:
from langchain_core.documents import Document

In [43]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader


dir_loader = DirectoryLoader("../data/text_files", glob="**/*.txt", loader_cls=TextLoader)
documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning is a branch of artificial intelligence that enables computers to learn patterns from data and \nmake predictions or decisions without being explicitly programmed. Instead of following fixed rules, machine learning models \nimprove their performance over time as they are exposed to more data. It is widely used in applications such as recommendation \nsystems, image recognition, fraud detection, speech recognition, and natural language processing. Due to the increasing \navailability of data and computational power, machine learning has become a core technology in modern AI systems.'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python is a high-level, easy-to-learn programming language widely used in web development, data science, artificial intelligence,\n automation, and software development. Created by Guido van Rossum, Python is known

In [44]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


dir_loader = DirectoryLoader("../data/pdf",
 glob="**/*.pdf", 
 loader_cls=PyMuPDFLoader,
)
pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': '3.0.26 (5.1.12)', 'creator': 'Microsoft® Word 2021', 'creationdate': '2025-08-07T12:06:15+05:30', 'source': '..\\data\\pdf\\abstract_final.pdf', 'file_path': '..\\data\\pdf\\abstract_final.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Garvita Sakhrani', 'subject': '', 'keywords': '', 'moddate': '2025-08-07T09:02:04+02:00', 'trapped': '', 'modDate': "D:20250807090204+02'00'", 'creationDate': "D:20250807120615+05'30'", 'page': 0}, page_content='Swami Keshvanand Institute of Technology, \nManagement & Gramothan, Jaipur \nDepartment of Information Technology ,Session 2024-25 \n \n \n \nThe National Rural Livelihoods Mission (NRLM) aims to empower rural communities by \norganizing individuals into Self Help Groups (SHGs) and enabling them to pursue \nentrepreneurial activities. However, despite producing a variety of products and \nservices, SHGs often face challenges such as restricted market access, low production \nvolumes, and lack

In [90]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(pdf_documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 160


In [63]:
type(pdf_documents[0])

langchain_core.documents.base.Document

### Embedding and vectorStoreDb

In [64]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List , Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [91]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> List[np.ndarray]:
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(texts)} texts.")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

# initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager 

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2168.24it/s]


Model loaded successfully.Embedding dimension: 384


C:\Users\user\AppData\Local\Temp\ipykernel_4076\1826839235.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully.Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [92]:
import os
class VectorStore:
    def __init__(self, collection_name: str = "document_embeddings", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialze_store()

    def _initialze_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name,
                metadata={"description": "Collection for document embeddings"})
            print(f"Vector store initialized with collection {self.collection_name} ")
            print(f"Existing document in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")

        print(f"Adding {len(documents)} documents to vector store.")

        ids=[]
        metadatas=[]
        document_texts=[]
        embeddings_list=[]

        for idx, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate unique id for each document
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)
            
            #prepare metadata
            metadata = dict(doc.metadata) if hasattr(doc, 'metadata') else {}
            metadata['doc_index'] = idx
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #document text
            document_texts.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding)

            try:
                self.collection.add(
                    ids=[doc_id],
                    embeddings=[embedding.tolist()],  # Convert numpy array to list for storage
                    metadatas=[metadata],
                    documents=[doc.page_content]
                )
                print(f"Successfully added document {len(documents)} documents to vector store.")
                print(f"Total documents in collection after addition: {self.collection.count()}")

            except Exception as e:
                print(f"Error adding document to vector store: {e}")
                raise



vector_store = VectorStore()
vector_store 

Vector store initialized with collection document_embeddings 
Existing document in collection: 744


In [93]:
chunks

[Document(metadata={'producer': '3.0.26 (5.1.12)', 'creator': 'Microsoft® Word 2021', 'creationdate': '2025-08-07T12:06:15+05:30', 'source': '..\\data\\pdf\\abstract_final.pdf', 'file_path': '..\\data\\pdf\\abstract_final.pdf', 'total_pages': 1, 'format': 'PDF 1.7', 'title': '', 'author': 'Garvita Sakhrani', 'subject': '', 'keywords': '', 'moddate': '2025-08-07T09:02:04+02:00', 'trapped': '', 'modDate': "D:20250807090204+02'00'", 'creationDate': "D:20250807120615+05'30'", 'page': 0}, page_content='Swami Keshvanand Institute of Technology, \nManagement & Gramothan, Jaipur \nDepartment of Information Technology ,Session 2024-25 \n \n \n \nThe National Rural Livelihoods Mission (NRLM) aims to empower rural communities by \norganizing individuals into Self Help Groups (SHGs) and enabling them to pursue \nentrepreneurial activities. However, despite producing a variety of products and \nservices, SHGs often face challenges such as restricted market access, low production \nvolumes, and lack

In [94]:
texts =[chunk.page_content for chunk in chunks]


#generate embeddings for the documents
embeddings = embedding_manager.generate_embeddings(texts)

#store in vectore database
vector_store.add_documents(chunks, embeddings)



Generating embeddings for 160 texts.


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:07<00:00,  1.43s/it]


Generated embeddings with shape: (160, 384)
Adding 160 documents to vector store.
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 745
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 746
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 747
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 748
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 749
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 750
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 751
Successfully added document 160 documents to vector store.
Total documents in collection after addition: 752
Successfully added document 160 documents to v

### RAG retriever pipeline from vector store

In [95]:
class RagRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5) -> List[Tuple[str, Dict[str, Any]]]:
        #generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #retrieve similar documents from vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"]
            )

            #process results
            retrieved_docs = []

            if results and results['documents']:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                score_threshold = 0.3
                for i, (doc_id,document, metadata, distance ) in enumerate(zip( ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:  # Filter based on similarity score threshold
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        
                        })

                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering) ")


                    #retrieved_docs.append((document, metadata))
                    #print(f"Rank {i+1}: Document ID: {doc_id}, Distance: {distance:.4f}, Metadata: {metadata}")
                    print(f"\nDocument Content:\n{document}")

            else:
                print("no documents found")
                return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

retriever = RagRetriever(vector_store, embedding_manager)
retriever

            

In [96]:
retriever.retrieve("What is Check sum?")

Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering) 

Document Content:
Sum 
11100010 
Checksum: 
Receiver site: 
00011101 
 
10101001 
00111001 
00011101 
Sum 
11111111 
Complement 
00000000 means that the pattern is O.K. 
 
Data Communication and Computer Networks 
 
INTERNET CHECKSUM 
Example 1: - The following two messages are sent 10101001, and 00111001.
Retrieved 1 documents (after filtering) 

Document Content:
Sum 
11100010 
Checksum: 
Receiver site: 
00011101 
 
10101001 
00111001 
00011101 
Sum 
11111111 
Complement 
00000000 means that the pattern is O.K. 
 
Data Communication and Computer Networks 
 
INTERNET CHECKSUM 
Example 1: - The following two messages are sent 10101001, and 00111001.  
Sender site: 
10101001 
00111001 
CS/ IV Sem
Retrieved 1 documents (after filtering) 

Document Content:
Sum 
11100010 
Checksum: 
Receiver site: 
00011101 
 
10101001 
00111001 
00011101 
Sum 
11111111 
Complement 
00000000 means that the pattern is O.K. 

In [97]:
retriever.retrieve("What is hamming code?")

Generating embeddings for 1 texts.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 19.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering) 

Document Content:
🞭 The hamming code uses the relation between redundancy bits and the data bits. 
🞭 Hamming Code can be applied to any number of data bits.
Retrieved 2 documents (after filtering) 

Document Content:
🞭 Hamming Code can be applied to any number of data bits.
Retrieved 3 documents (after filtering) 

Document Content:
also corrects it. 
🞭 Hamming Code uses a number of parity bits located at certain positions in the 
codeword. 
🞭 The number of parity bits depends upon the number of information bits. 
🞭 The hamming code uses the relation between redundancy bits and the data bits.
Retrieved 4 documents (after filtering) 

Document Content:
HAMMING 
CODE 
 
Data Communication and Computer Networks 
CS/ IV Sem 
Algorithm of Hamming code: 
🞭 An information of 'd' bits are added to the redundant bits 'r' to form  
d+r. 
🞭 The location of each of the (d+r) digits is assigned a decimal value.
Retr

In [89]:
retriever.retrieve("What is the main idea of unit 2 ?")

Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 42.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering) 

Document Content:
tools and technologies. 
V2: Be a leading department in the state and region by 
imparting in depth knowledge to  
the students in emerging technologies in computer 
science & engineering.                           
Mission of CSE department is to: 
Deliver resources in IT enable domain through:
Retrieved 0 documents (after filtering) 

Document Content:
UNIT-2 
 
Data Link Layer 
Data Communication and Computer Networks 
CS/ IV Sem
Retrieved 0 documents (after filtering) 

Document Content:
UNIT-2 
 
Data Link Layer 
Data Communication and Computer Networks 
CS/ IV Sem
Retrieved 0 documents (after filtering) 

Document Content:
UNIT-2 
 
Data Link Layer 
Data Communication and Computer Networks 
CS/ IV Sem
Retrieved 0 documents (after filtering) 

Document Content:
applying new ideas. 
PEO2: Prepared to be responsible professionals in their domain of interest. 
PEO3: Able to apply the

In [99]:
import anthropic

def answer_question(question: str, retriever: RagRetriever) -> str:
    # Step 1: retrieve relevant chunks
    retrieved_docs = retriever.retrieve(question, top_k=5)
    
    if not retrieved_docs:
        return "No relevant documents found."

    # Step 2: build context from chunks
    context = "\n\n".join([doc['content'] for doc in retrieved_docs])

    # Step 3: call LLM with context + question
    client = anthropic.Anthropic()
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        messages=[{
            "role": "user",
            "content": f"""Answer the question using only the context below.
            
Context:
{context}

Question: {question}"""
        }]
    )
    return response.content[0].text

# Usage
answer = answer_question("What is a checksum?", retriever)
print(answer)

Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 86.89it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering) 

Document Content:
INTERNET CHECKSUM 
🞭 Several Internet protocols (e.g. IP, TCP, UDP) use check bits to detect errors in the 
IP header (or in the header and data for TCP/UDP) 
🞭 A checksum is calculated for header contents and included in a special field.
Retrieved 0 documents (after filtering) 

Document Content:
Sum 
11100010 
Checksum: 
Receiver site: 
00011101 
 
10101001 
00111001 
00011101 
Sum 
11111111 
Complement 
00000000 means that the pattern is O.K. 
 
Data Communication and Computer Networks 
 
INTERNET CHECKSUM 
Example 1: - The following two messages are sent 10101001, and 00111001.
Retrieved 0 documents (after filtering) 

Document Content:
Sum 
11100010 
Checksum: 
Receiver site: 
00011101 
 
10101001 
00111001 
00011101 
Sum 
11111111 
Complement 
00000000 means that the pattern is O.K. 
 
Data Communication and Computer Networks 
 
INTERNET CHECKSUM 
Example 1: - The following two m